# GeoSat Two-Batch Processing (Colab Cloud GPU)

This notebook runs the land-cover change-detection pipeline from GitHub, uses the trained model and Google Earth Engine service key stored in Google Drive, and syncs each completed city back to Drive.

### Run order
1. Put `models/resnet50_best.pt` and `service_account.json` in the Drive folder configured in cell 2.
2. Run all cells with `RUN_MODE = "smoke"`. The default smoke test is `TinyChennai`.
3. Confirm that `TinyChennai` finishes and its outputs appear in Drive.
4. Change `RUN_MODE` to `"full"`, choose `CITY_BATCH = 1` or `CITY_BATCH = 2`, keep `FORCE_RERUN = False`, and run the validation and processing cells again.
5. Batch 1 processes the six Tamil Nadu cities. Batch 2 processes the remaining production cities.

### Important details
- `64x64` is the model inference patch size, not the city size.
- City bounding boxes are downloaded at 10 m resolution and split recursively when an Earth Engine request exceeds the 50 MB limit.
- The service key remains in Drive and is referenced through `GOOGLE_APPLICATION_CREDENTIALS`; it is not copied into the GitHub checkout.
- Update `DRIVE_ROOT` in cell 2 if your Drive folder has a different name or location.
- With `FORCE_RERUN = False`, a city is skipped when its Drive output folder already contains `completed.json`.
- Set `FORCE_RERUN = True` only when you intentionally want to regenerate completed cities.

### Prerequisites
1. Enable a GPU runtime in Colab, preferably a T4 or better.
2. Ensure the GitHub repository URL in cell 3 is reachable.
3. Ensure the Drive model and service-key paths in cell 2 exist.

In [ ]:
# 1. Mount Google Drive & Check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print("\nCUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable a GPU runtime in Google Colab (Runtime > Change runtime type).")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Configuration
# ---------------------------------------------------------
# Keep the service key in Drive. It is never copied into the cloned repository.
DRIVE_ROOT = "/content/drive/MyDrive/LandCoverChangeResults"
MODEL_PATH = f"{DRIVE_ROOT}/models/resnet50_best.pt"
DRIVE_GEE_CREDENTIALS = f"{DRIVE_ROOT}/service_account.json"

START_YEAR = 2019
END_YEAR = 2023
BATCH_SIZE = 32
FORCE_RERUN = False

# Run "smoke" first. After TinyChennai succeeds, change to "full".
RUN_MODE = "smoke"  # "smoke" or "full"
SMOKE_CITY = "TinyChennai"

# Full mode is split into two batches:
# 1 = Tamil Nadu cities, 2 = all other production cities.
CITY_BATCH = 1

# 64x64 is only the model patch size. Geographic city downloads are much larger
# and are split into smaller Earth Engine requests when a request exceeds 50 MB.
MODEL_PATCH_SIZE = 64
PATCH_RESOLUTION_METERS = 10

# ---------------------------------------------------------

In [ ]:
# 3. Setup Environment & Clone Repository
import os
import shutil
import sys

!pip install -q earthengine-api torch torchvision rasterio pandas python-dotenv pyproj pillow

REPO_URL = "https://github.com/saran318/Geo_Sat.git"
REPO_DIR = "/content/Geo_Sat"
if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"

os.chdir(os.path.join(REPO_DIR, "geo_sat project"))

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Trained model not found at {MODEL_PATH}")
os.makedirs("models", exist_ok=True)
shutil.copy2(MODEL_PATH, "models/resnet50_best.pt")

if not os.path.isfile(DRIVE_GEE_CREDENTIALS):
    raise FileNotFoundError(f"GEE service key not found at {DRIVE_GEE_CREDENTIALS}")
# The key stays in Drive; the pipeline reads this absolute path from the environment.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = DRIVE_GEE_CREDENTIALS
print("Repository, model, and Drive credential path are ready.")

In [ ]:
# 4. Validate Cities and Select the Run Set
import json
import time
import datetime
import gc
import subprocess
import math

with open('config/cities.json', 'r') as f:
    ALL_CITIES_CONFIG = json.load(f)

BATCH_1_CITIES = [
    "Coimbatore",
    "Madurai",
    "Tiruchirappalli",
    "Salem",
    "Kanchipuram",
    "Chennai",
]
BATCH_2_CITIES = [
    city for city in ALL_CITIES_CONFIG
    if not city.startswith("Tiny") and city not in BATCH_1_CITIES
]

if RUN_MODE == "smoke":
    target_cities = [SMOKE_CITY]
elif RUN_MODE == "full":
    if CITY_BATCH == 1:
        target_cities = BATCH_1_CITIES
    elif CITY_BATCH == 2:
        target_cities = BATCH_2_CITIES
    else:
        raise ValueError("CITY_BATCH must be 1 (Tamil Nadu) or 2 (other cities).")
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")

unknown_cities = [city for city in target_cities if city not in ALL_CITIES_CONFIG]
if unknown_cities:
    raise ValueError(f"Unknown cities: {unknown_cities}")

if START_YEAR == END_YEAR:
    raise ValueError("START_YEAR and END_YEAR must be different.")

for city in target_cities:
    min_lon, min_lat, max_lon, max_lat = ALL_CITIES_CONFIG[city]["bbox"]
    center_lat = (min_lat + max_lat) / 2
    width_m = (max_lon - min_lon) * 111320 * math.cos(math.radians(center_lat))
    height_m = (max_lat - min_lat) * 110540
    print(f"{city}: bbox {width_m:.0f}m x {height_m:.0f}m; model patches are {MODEL_PATCH_SIZE}x{MODEL_PATCH_SIZE} pixels at {PATCH_RESOLUTION_METERS}m")
    if city.startswith("Tiny") and (width_m <= MODEL_PATCH_SIZE * PATCH_RESOLUTION_METERS or height_m <= MODEL_PATCH_SIZE * PATCH_RESOLUTION_METERS):
        raise ValueError(f"{city} bbox is too small for the configured model patch size.")

os.makedirs(DRIVE_ROOT, exist_ok=True)
status_file = f"{DRIVE_ROOT}/processing_status.json"
if os.path.exists(status_file):
    with open(status_file, 'r') as f:
        processing_status = json.load(f)
else:
    processing_status = {}

print(f"Run mode: {RUN_MODE}")
print(f"City batch: {CITY_BATCH if RUN_MODE == 'full' else 'smoke'}")
print(f"Target cities ({len(target_cities)}): {target_cities}")
print("Earth Engine requests are adaptively subdivided when they exceed the 50 MB limit.")

In [ ]:
# 5. Execute Pipeline Sequentially
for city in target_cities:
    print(f"\n{'='*50}")
    print(f"Processing City: {city}")
    print(f"{'='*50}")

    drive_city_dir = f"{DRIVE_ROOT}/{city}"
    drive_completed = f"{drive_city_dir}/completed.json"

    if not FORCE_RERUN and os.path.exists(drive_completed):
        print(f"City {city} already completed. Skipping.")
        continue

    processing_status[city] = {
        "status": "running",
        "start_time": datetime.datetime.now().isoformat()
    }
    with open(status_file, 'w') as f:
        json.dump(processing_status, f, indent=4)

    start_time = time.time()
    try:
        cmd = [
            sys.executable,
            "run_real_pipeline.py",
            "--city", city,
            "--years", str(START_YEAR), str(END_YEAR),
            "--batch_size", str(BATCH_SIZE),
        ]
        # Stream pipeline logs live so long downloads/inference are visibly active.
        result = subprocess.run(cmd, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Pipeline failed for {city} (exit {result.returncode}).")

        duration = time.time() - start_time
        local_results = f"data/results/{city}"
        if not os.path.exists(local_results):
            raise FileNotFoundError(f"Local results not found at {local_results}")

        os.makedirs(drive_city_dir, exist_ok=True)
        for item in os.listdir(local_results):
            source = os.path.join(local_results, item)
            destination = os.path.join(drive_city_dir, item)
            if os.path.isdir(source):
                if os.path.exists(destination):
                    shutil.rmtree(destination)
                shutil.copytree(source, destination)
            else:
                shutil.copy2(source, destination)

        processing_status[city].update({
            "status": "completed",
            "end_time": datetime.datetime.now().isoformat(),
            "duration_sec": round(duration, 2),
            "output_path": drive_city_dir
        })
        print(f"Successfully processed and synced {city} in {duration:.2f}s")

    except Exception as e:
        print(f"ERROR processing {city}: {e}")
        processing_status[city].update({
            "status": "failed",
            "end_time": datetime.datetime.now().isoformat(),
            "error": str(e)
        })

    with open(status_file, 'w') as f:
        json.dump(processing_status, f, indent=4)

    torch.cuda.empty_cache()
    gc.collect()
    if os.path.exists(f"data/results/{city}"):
        shutil.rmtree(f"data/results/{city}")

In [ ]:
# 6. Final Summary
print("\n" + "*"*60)
print("CURRENT RUN SUMMARY")
print("*"*60)
for city in target_cities:
    stat = processing_status.get(city, {})
    status = stat.get('status', 'unknown')
    duration = stat.get('duration_sec', 'N/A')
    error = stat.get('error', '')
    print(f"{city:15s} | {status:10s} | {duration}s | {error}")
print("*"*60)
print("Historical statuses remain in processing_status.json, but are not shown above.")